# 90 — TIC–TIM Emprego e Estrutura Econômica: pipeline reprodutível completo

Este notebook integra aquisição, validação, transformação analítica e geração de tabelas, gráficos e mapas para o projeto TIC–TIM. Ele foi desenhado para operar sobre o repositório `sou_rais`, evitando dependência de Google Drive/Colab e usando caminhos relativos.

**Escopo coberto**

- RAIS vínculos e estabelecimentos;
- Novo CAGED;
- snapshots do CNPJ;
- indicadores municipais e regionais;
- estrutura CNAE, ocupacional e sociodemográfica;
- remuneração e escolaridade;
- concentração e especialização;
- gráficos regionais e municipais;
- mapas coropléticos municipais;
- exportação padronizada de tabelas e figuras;
- manifesto de execução e hashes dos produtos.

**Observação metodológica:** RAIS, Novo CAGED e CNPJ têm unidades estatísticas e temporalidades distintas. O notebook não concatena essas bases em uma série única.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json, hashlib, math, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
DADOS = ROOT / "dados"
PROC = DADOS / "processado"
SAIDA = DADOS / "analise_tic_tim"
TAB = SAIDA / "tabelas"
FIG = SAIDA / "figuras"
MAP = SAIDA / "mapas"
CTRL = SAIDA / "controle"
for p in [SAIDA, TAB, FIG, MAP, CTRL]: p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('SAIDA:', SAIDA)

## 1. Configuração e aquisição

O notebook reutiliza a configuração do repositório e, quando solicitado, chama a própria CLI para planejar/baixar as bases. Isso mantém a aquisição idêntica ao pipeline principal do projeto.

In [ ]:
from sou_rais import carregar_config
cfg = carregar_config(ROOT)
print('Municípios:', len(cfg.municipios))
print('Período RAIS:', cfg.ano_inicial, cfg.ano_final)
print('Período CAGED:', cfg.competencia_inicial, cfg.competencia_final)
print('Snapshots CNPJ:', cfg.snapshot_inicial, cfg.snapshot_final)

In [ ]:
# Opcional: descomente para executar a aquisição completa pela CLI.
# !sou-rais doctor
# !sou-rais plan
# !sou-rais download all
# !sou-rais validate

print('Use as quatro linhas acima quando quiser reproduzir a aquisição desde a fonte.')

## 2. Descoberta dos Parquets locais

In [ ]:
def ler_parquets(pasta: Path) -> pd.DataFrame:
    arquivos = sorted(pasta.rglob('*.parquet')) if pasta.exists() else []
    if not arquivos:
        return pd.DataFrame()
    dfs=[]
    for f in arquivos:
        try:
            d=pd.read_parquet(f)
            d['_arquivo_fonte']=str(f.relative_to(ROOT))
            dfs.append(d)
        except Exception as e:
            print('Falha:', f, e)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

rais_v = ler_parquets(PROC/'rais'/'vinculos')
rais_e = ler_parquets(PROC/'rais'/'estabelecimentos')
caged = ler_parquets(PROC/'caged')
cnpj = ler_parquets(PROC/'cnpj')
for nome,df in [('RAIS vínculos',rais_v),('RAIS estabelecimentos',rais_e),('Novo CAGED',caged),('CNPJ',cnpj)]:
    print(nome, df.shape)

## 3. Utilitários de harmonização

Como os esquemas públicos podem evoluir, o notebook usa resolução defensiva de nomes de colunas. Toda variável usada em indicador é explicitamente registrada.

In [ ]:
def primeira_coluna(df, candidatos):
    for c in candidatos:
        if c in df.columns:
            return c
    return None

def padronizar_municipio(df):
    if df.empty: return df
    c=primeira_coluna(df,['id_municipio','municipio','cod_municipio','codigo_municipio'])
    if c and c!='id_municipio': df=df.rename(columns={c:'id_municipio'})
    if 'id_municipio' in df: df['id_municipio']=df['id_municipio'].astype(str).str.zfill(7)
    return df

rais_v=padronizar_municipio(rais_v); rais_e=padronizar_municipio(rais_e); caged=padronizar_municipio(caged); cnpj=padronizar_municipio(cnpj)

COLS={
'ano': ['ano','ano_rais'],
'competencia':['ano_mes','competencia','mes'],
'cnae':['cnae_2','cnae_2_subclasse','cnae_2_classe','cnae'],
'cbo':['cbo_2002','cbo_2002_familia','cbo'],
'remun':['remuneracao_media','valor_remuneracao_media','remuneracao_dezembro_nominal','salario_medio'],
'sexo':['sexo'], 'idade':['idade'], 'escolaridade':['escolaridade'], 'raca':['raca_cor','raca'],
'vinculo_id':['id_vinculo','pis','cpf','id_trabalhador'],
'estab_id':['id_estabelecimento','cnpj','cnpj_basico'],
'saldo':['saldo_movimentacao','saldo'],
'admissoes':['admissoes','admitidos'], 'desligamentos':['desligamentos','desligados']
}

def col(df,k): return primeira_coluna(df,COLS[k])
print({k:col(rais_v,k) for k in ['ano','cnae','cbo','remun','sexo','idade','escolaridade','raca','estab_id']})

## 4. Estoque formal e trajetória municipal/regional

In [ ]:
def tabela_estoque_rais(df):
    if df.empty: return pd.DataFrame()
    ano=col(df,'ano');
    if not ano: raise KeyError('Ano RAIS não encontrado')
    g=df.groupby(['id_municipio',ano],dropna=False).size().rename('estoque').reset_index().rename(columns={ano:'ano'})
    return g

estoque=tabela_estoque_rais(rais_v)
if not estoque.empty:
    estoque.to_csv(TAB/'01_estoque_municipio_ano.csv',index=False)
    reg=estoque.groupby('ano',as_index=False)['estoque'].sum()
    reg['variacao_pct']=reg['estoque'].pct_change()*100
    reg.to_csv(TAB/'02_estoque_regional_ano.csv',index=False)
    display(reg.tail())

In [ ]:
if not estoque.empty:
    reg=estoque.groupby('ano',as_index=False)['estoque'].sum()
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(reg['ano'],reg['estoque'],marker='o')
    ax.set(title='Evolução do estoque de vínculos formais — conjunto municipal',xlabel='Ano',ylabel='Vínculos')
    ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(FIG/'01_evolucao_estoque_regional.png',dpi=180); plt.show()

## 5. Crescimento absoluto e contribuição municipal

In [ ]:
if not estoque.empty:
    a0,a1=estoque['ano'].min(),estoque['ano'].max()
    p=estoque.pivot(index='id_municipio',columns='ano',values='estoque').fillna(0)
    cresc=pd.DataFrame({'id_municipio':p.index,'estoque_inicial':p.get(a0,0),'estoque_final':p.get(a1,0)}).reset_index(drop=True)
    cresc['acrescimo']=cresc['estoque_final']-cresc['estoque_inicial']
    total=cresc['acrescimo'].sum()
    cresc['contribuicao_pct']=np.where(total!=0,100*cresc['acrescimo']/total,np.nan)
    cresc=cresc.sort_values('acrescimo',ascending=False)
    cresc.to_csv(TAB/'03_crescimento_municipal.csv',index=False)
    fig,ax=plt.subplots(figsize=(10,7)); top=cresc.head(15).sort_values('acrescimo')
    ax.barh(top['id_municipio'],top['acrescimo']); ax.set(title=f'Maiores acréscimos absolutos de vínculos — {a0}–{a1}',xlabel='Vínculos')
    fig.tight_layout(); fig.savefig(FIG/'02_maiores_acrescimos_municipais.png',dpi=180); plt.show()

## 6. Estrutura econômica, QL, HHI e mudança setorial

In [ ]:
def setor2(x):
    s=str(x)
    dig=''.join(ch for ch in s if ch.isdigit())
    return dig[:2] if len(dig)>=2 else s

if not rais_v.empty and col(rais_v,'cnae') and col(rais_v,'ano'):
    d=rais_v[['id_municipio',col(rais_v,'ano'),col(rais_v,'cnae')]].copy()
    d.columns=['id_municipio','ano','cnae']; d['setor']=d['cnae'].map(setor2)
    sec=d.groupby(['id_municipio','ano','setor']).size().rename('vinculos').reset_index()
    tot_m=sec.groupby(['id_municipio','ano'])['vinculos'].transform('sum')
    sec['share_mun']=sec['vinculos']/tot_m
    reg=sec.groupby(['ano','setor'],as_index=False)['vinculos'].sum(); reg['share_reg']=reg['vinculos']/reg.groupby('ano')['vinculos'].transform('sum')
    sec=sec.merge(reg[['ano','setor','share_reg']],on=['ano','setor'],how='left'); sec['ql']=sec['share_mun']/sec['share_reg']
    hhi=sec.assign(sq=sec['share_mun']**2).groupby(['id_municipio','ano'],as_index=False)['sq'].sum().rename(columns={'sq':'hhi'})
    sec.to_csv(TAB/'04_estrutura_setorial_ql.csv',index=False); hhi.to_csv(TAB/'05_hhi_setorial.csv',index=False)
    display(sec.head())

In [ ]:
if 'sec' in globals() and not sec.empty:
    a0,a1=sec['ano'].min(),sec['ano'].max()
    x=sec.groupby(['ano','setor'],as_index=False)['vinculos'].sum().pivot(index='setor',columns='ano',values='vinculos').fillna(0)
    x['delta']=x.get(a1,0)-x.get(a0,0); top=x.sort_values('delta',ascending=False).head(15).sort_values('delta')
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(top.index,top['delta']); ax.set(title=f'Contribuição setorial ao crescimento líquido — {a0}–{a1}',xlabel='Vínculos')
    fig.tight_layout(); fig.savefig(FIG/'03_contribuicao_setorial_crescimento.png',dpi=180); plt.show()

## 7. Estrutura ocupacional

In [ ]:
if not rais_v.empty and col(rais_v,'cbo') and col(rais_v,'ano'):
    d=rais_v[['id_municipio',col(rais_v,'ano'),col(rais_v,'cbo')]].copy(); d.columns=['id_municipio','ano','cbo']; d['cbo']=d['cbo'].astype(str)
    occ=d.groupby(['id_municipio','ano','cbo']).size().rename('vinculos').reset_index()
    occ.to_csv(TAB/'06_estrutura_ocupacional.csv',index=False)
    a0,a1=occ['ano'].min(),occ['ano'].max(); x=occ.groupby(['ano','cbo'])['vinculos'].sum().unstack('ano',fill_value=0); x['delta']=x.get(a1,0)-x.get(a0,0)
    top=x.sort_values('delta',ascending=False).head(15).sort_values('delta')
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(top.index,top['delta']); ax.set(title=f'Acréscimo de vínculos por ocupação — {a0}–{a1}',xlabel='Vínculos')
    fig.tight_layout(); fig.savefig(FIG/'04_crescimento_ocupacional.png',dpi=180); plt.show()

## 8. Escolaridade, remuneração e perfil sociodemográfico

In [ ]:
def resumo_categoria(df, chave, nome_saida):
    c=col(df,chave); a=col(df,'ano')
    if df.empty or not c or not a: return pd.DataFrame()
    g=df.groupby(['id_municipio',a,c]).size().rename('vinculos').reset_index().rename(columns={a:'ano',c:chave})
    g.to_csv(TAB/nome_saida,index=False); return g

esc=resumo_categoria(rais_v,'escolaridade','07_escolaridade.csv')
sexo=resumo_categoria(rais_v,'sexo','08_sexo.csv')
raca=resumo_categoria(rais_v,'raca','09_raca_cor.csv')

idadec=col(rais_v,'idade'); anoc=col(rais_v,'ano')
if idadec and anoc:
    d=rais_v[['id_municipio',anoc,idadec]].copy(); d.columns=['id_municipio','ano','idade']; d['idade']=pd.to_numeric(d['idade'],errors='coerce')
    bins=[0,17,24,29,39,49,59,200]; labels=['<=17','18-24','25-29','30-39','40-49','50-59','60+']
    d['faixa_etaria']=pd.cut(d['idade'],bins=bins,labels=labels)
    et=d.groupby(['id_municipio','ano','faixa_etaria'],observed=True).size().rename('vinculos').reset_index(); et.to_csv(TAB/'10_faixa_etaria.csv',index=False)

In [ ]:
rem=col(rais_v,'remun'); ano=col(rais_v,'ano')
if rem and ano:
    d=rais_v[['id_municipio',ano,rem]].copy(); d.columns=['id_municipio','ano','remuneracao']; d['remuneracao']=pd.to_numeric(d['remuneracao'],errors='coerce')
    r=d.groupby(['id_municipio','ano'])['remuneracao'].agg(['count','mean','median']).reset_index()
    r.to_csv(TAB/'11_remuneracao_municipal.csv',index=False); display(r.tail())

## 9. Novo CAGED — fluxos

In [ ]:
if not caged.empty:
    comp=col(caged,'competencia'); saldo=col(caged,'saldo'); adm=col(caged,'admissoes'); desl=col(caged,'desligamentos')
    if comp:
        gcols=['id_municipio',comp]
        if saldo:
            fl=caged.groupby(gcols,as_index=False)[saldo].sum().rename(columns={comp:'competencia',saldo:'saldo'})
        else:
            # Em microdados de movimentação, quando houver indicador de admissão/desligamento, adaptar aqui explicitamente.
            fl=caged.groupby(gcols).size().rename('movimentacoes').reset_index().rename(columns={comp:'competencia'})
        fl.to_csv(TAB/'12_novo_caged_fluxos.csv',index=False); display(fl.head())

## 10. Estabelecimentos e concentração de empregadores

In [ ]:
if not rais_v.empty and col(rais_v,'estab_id') and col(rais_v,'ano'):
    ec=col(rais_v,'estab_id'); ac=col(rais_v,'ano')
    g=rais_v.groupby(['id_municipio',ac,ec]).size().rename('vinculos').reset_index(); g.columns=['id_municipio','ano','estabelecimento','vinculos']
    def top10_share(x): return x.nlargest(10,'vinculos')['vinculos'].sum()/x['vinculos'].sum() if x['vinculos'].sum() else np.nan
    conc=g.groupby(['id_municipio','ano']).apply(top10_share,include_groups=False).rename('share_top10').reset_index()
    conc.to_csv(TAB/'13_concentracao_top10_empregadores.csv',index=False)
    ano_max=conc['ano'].max(); top=conc.query('ano==@ano_max').sort_values('share_top10',ascending=False).head(15).sort_values('share_top10')
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(top['id_municipio'],100*top['share_top10']); ax.set(title=f'Participação dos dez maiores empregadores — {ano_max}',xlabel='% dos vínculos')
    fig.tight_layout(); fig.savefig(FIG/'07_top10_empregadores.png',dpi=180); plt.show()

## 11. CNPJ — fotografia cadastral

A saída abaixo é deliberadamente tratada como **fotografia cadastral**, e não como série de nascimentos/mortes empresariais.

In [ ]:
if not cnpj.empty and 'id_municipio' in cnpj:
    cols=['id_municipio']
    for c in ['data','snapshot','data_extracao','opcao_simples','opcao_mei','situacao_cadastral','cnpj_basico']:
        if c in cnpj: cols.append(c)
    base=cnpj[cols].copy()
    snap=primeira_coluna(base,['snapshot','data','data_extracao'])
    keys=['id_municipio']+([snap] if snap else [])
    resumo=base.groupby(keys).size().rename('cnpjs').reset_index()
    resumo.to_csv(TAB/'14_cnpj_fotografia_cadastral.csv',index=False)

## 12. Mapas municipais

Se `geopandas` estiver instalado, o notebook baixa/usa uma malha municipal informada em `MALHA_MUNICIPAL` ou tenta ler `dados/auxiliares/municipios.geojson`. Para reprodutibilidade estrita, recomenda-se versionar apenas o script e registrar URL, data e hash da malha usada.

In [ ]:
try:
    import geopandas as gpd
    GEO=Path(os.getenv('MALHA_MUNICIPAL', DADOS/'auxiliares'/'municipios.geojson'))
    if GEO.exists() and not estoque.empty:
        geo=gpd.read_file(GEO)
        cod=primeira_coluna(geo,['id_municipio','CD_MUN','GEOCODIGO','code_muni'])
        if cod:
            geo['id_municipio']=geo[cod].astype(str).str.zfill(7)
            ano_max=estoque['ano'].max(); z=estoque.query('ano==@ano_max')
            m=geo.merge(z,on='id_municipio',how='inner')
            fig,ax=plt.subplots(figsize=(9,9)); m.plot(column='estoque',legend=True,ax=ax); ax.set_axis_off(); ax.set_title(f'Estoque de vínculos formais — {ano_max}')
            fig.tight_layout(); fig.savefig(MAP/'01_estoque_municipal.png',dpi=180); plt.show()
    else:
        print('Malha municipal não encontrada. Defina MALHA_MUNICIPAL ou salve dados/auxiliares/municipios.geojson')
except ImportError:
    print('geopandas não instalado. Instale o extra de análise geoespacial para gerar mapas.')

## 13. Tabelas municipais para fichas

In [ ]:
if not estoque.empty:
    ano_max=estoque['ano'].max(); ano_min=estoque['ano'].min()
    p=estoque.pivot(index='id_municipio',columns='ano',values='estoque')
    ficha=pd.DataFrame(index=p.index)
    ficha['estoque_inicial']=p.get(ano_min); ficha['estoque_final']=p.get(ano_max); ficha['variacao_pct']=100*(ficha['estoque_final']/ficha['estoque_inicial']-1)
    if 'hhi' in globals(): ficha=ficha.join(hhi.query('ano==@ano_max').set_index('id_municipio')['hhi'])
    if 'conc' in globals(): ficha=ficha.join(conc.query('ano==@ano_max').set_index('id_municipio')['share_top10'])
    ficha.reset_index().to_csv(TAB/'15_quadro_sintese_fichas_municipais.csv',index=False)
    display(ficha.head())

## 14. Auditoria de completude

In [ ]:
esperados=[
'01_estoque_municipio_ano.csv','02_estoque_regional_ano.csv','03_crescimento_municipal.csv',
'04_estrutura_setorial_ql.csv','05_hhi_setorial.csv','06_estrutura_ocupacional.csv',
'07_escolaridade.csv','08_sexo.csv','09_raca_cor.csv','10_faixa_etaria.csv',
'11_remuneracao_municipal.csv','12_novo_caged_fluxos.csv','13_concentracao_top10_empregadores.csv',
'14_cnpj_fotografia_cadastral.csv','15_quadro_sintese_fichas_municipais.csv']
status=pd.DataFrame({'arquivo':esperados}); status['existe']=status['arquivo'].map(lambda x:(TAB/x).exists())
status.to_csv(CTRL/'auditoria_completude.csv',index=False); display(status)

## 15. Manifesto e hashes dos produtos

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

rows=[]
for pasta in [TAB,FIG,MAP,CTRL]:
    for f in sorted(pasta.glob('*')):
        if f.is_file(): rows.append({'arquivo':str(f.relative_to(ROOT)),'bytes':f.stat().st_size,'sha256':sha256(f)})
manifest=pd.DataFrame(rows); manifest.to_csv(CTRL/'manifesto_produtos.csv',index=False)
meta={'municipios':cfg.municipios,'raiz':str(ROOT),'produtos':len(rows),'observacao':'RAIS, Novo CAGED e CNPJ tratados como fontes distintas.'}
(CTRL/'metadados_execucao.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
display(manifest.tail())

## 16. Critérios de interpretação

1. **RAIS** é estoque anual de vínculos declarados; mudança entre anos não equivale a fluxo bruto.
2. **Novo CAGED** representa movimentações mensais desde 2020; não deve ser emendado mecanicamente à RAIS ou ao CAGED antigo.
3. **CNPJ** é fotografia cadastral; snapshots não são uma série de emprego nem uma série de abertura/fechamento sem tratamento adicional.
4. **QL** mede especialização relativa, não competitividade.
5. **HHI** mede concentração, não vulnerabilidade automática.
6. Indicadores de sexo, raça/cor, idade, escolaridade e remuneração dependem de cobertura e preenchimento das variáveis; sempre auditar não informados.
7. Mapas municipais são descritivos na escala administrativa. Territorialização intraurbana de empregadores exige pipeline específico (CNEFE/endereço/geocodificação), que deve ser mantido como módulo separado para não confundir precisão municipal com localização postal.

Ao final da execução, os produtos canônicos desta análise ficam em `dados/analise_tic_tim/`.